# Notebook 03 — Pipeline v3: Granularidad Cuenca + PU-Learning

**Objetivo:** Implementar el pipeline de Fase 2 con granularidad de cuenca hidrográfica
(HydroBASINS nivel 10, ~549 cuencas en Antioquia) y comparar clasificación binaria clásica
contra PU-Learning (`BaggingPuClassifier`).

**Configuración:** `configs/antioquia_deslizamientos.yaml` (v3.0.0)
**Depende de:** `01_antioquia_deslizamientos_scope.ipynb` (dataset UNGRD)

## ¿Por qué cambiar la granularidad?

En el Notebook 02, el dataset departamental (204 semanas × Antioquia entera) generaba
etiquetas con 64% de positivos — casi siempre había algún deslizamiento en los 63.000 km²
de Antioquia. Esto hace el problema trivial y no permite aprender señales reales.

Con cuencas de nivel 10 (~100-500 km²), el dataset tiene:
- ~114.000 instancias (549 cuencas × 209 semanas)
- ~0.4% de positivos — desbalance real 1:250
- Señal espacial: la cuenca donde ocurre el evento es física y geomorfológicamente relevante

## ¿Qué es PU-Learning y por qué aquí?

Nuestros datos son **Positive-Unlabeled (PU)**:
- Clase 1 = eventos UNGRD reportados (presencias confirmadas)
- Clase 0 = semanas sin reporte (mezcla de verdaderos negativos + falsos negativos)

UNGRD no registra eventos sin víctimas o infraestructura afectada. Muchas semanas/cuencas
etiquetadas como 0 pueden haber tenido deslizamientos no reportados.

`BaggingPuClassifier` (Mordelet & Vert, 2014) maneja esto tratando la clase 0 como
'unlabeled' en lugar de 'confirmed negative'.

## Estructura de este notebook

```
Pilar 1 — Datos espaciales y construccion del dataset v3
    Descargar/cargar HydroBASINS nivel 10
    Geocodificar eventos UNGRD con centroides municipales
    Asignar eventos a cuencas por interseccion espacial
    Construir grid (semana x cuenca) + pseudo-ausencias

Pilar 2 — Experimentacion con validacion temporal de panel
    CV adaptado para datos de panel (semana x cuenca)
    Comparar: Dummy / LogisticReg / RandomForest / BaggingPU
    Registrar todo en MLflow

Pilar 3 — Model Registry
    Registrar el mejor modelo
    Comparar AUC v2 (departamental) vs v3 (cuenca)
```

> **Principio guia:** la mejora de granularidad espacial es tan importante
> como la mejora de la fuente de precipitacion.

## 0. Entorno y configuración

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
for _p in [_cwd, *_cwd.parents]:
    if (_p / "pyproject.toml").exists():
        _root = _p
        break
else:
    _root = _cwd.parent

sys.path.insert(0, str(_root / "src"))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

from experiment.config import load_config
from experiment.spatial import (
    download_hydrobasins,
    download_municipio_centroids,
    get_ungrd_with_coords,
    assign_events_to_cuencas,
    build_event_grid,
    generate_pseudo_absences,
)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11
sns.set_theme(style="whitegrid")

cfg = load_config(project_root=_root)

print("Librerias cargadas")
print(f"  Experimento MLflow : {cfg.mlflow.experiment_name}")
print(f"  HydroSHEDS nivel   : {cfg.espacial.hydrobasins_nivel}")
print(f"  Pseudo-absence P   : precip={cfg.espacial.pseudo_absence.precip_percentil} area={cfg.espacial.pseudo_absence.area_percentil}")

---
## Pilar 1 — Datos espaciales y construcción del dataset v3

### 1.1 Inicializar MLflow

In [ ]:
mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
experiment = mlflow.set_experiment(cfg.mlflow.experiment_name)

print(f"MLflow configurado")
print(f"  Experimento ID : {experiment.experiment_id}")
print(f"  URI            : {cfg.mlflow_tracking_uri}")
print(f"\nPara la UI: mlflow ui --backend-store-uri {cfg.mlflow_tracking_uri.replace('sqlite:///', 'sqlite:///').replace('sqlite:///', '')}")

### 1.2 Cargar capas espaciales

HydroBASINS nivel 10 (~100-500 km² por cuenca) y centroides municipales de Antioquia.
Si los archivos no existen localmente, se descargan automáticamente.

In [ ]:
nivel = cfg.espacial.hydrobasins_nivel  # 10 en config v3.0.0

gdf_cuencas = download_hydrobasins(nivel=nivel)
gdf_mpios   = download_municipio_centroids()

print(f"Cuencas HydroBASINS lev{nivel:02d}: {len(gdf_cuencas)}")
print(f"Municipios Antioquia    : {len(gdf_mpios)}")
print(f"Columnas cuencas        : {[c for c in gdf_cuencas.columns if c != 'geometry']}")
print()
print("Estadisticas cuencas:")
print(gdf_cuencas[["SUB_AREA", "UP_AREA", "DIST_MAIN", "ORDER"]].describe().round(1).to_string())

### 1.3 Geocodificar eventos UNGRD y asignar a cuencas

UNGRD tiene fechas + municipio pero no coordenadas individuales.
Usamos centroides municipales (GADM nivel 2) como aproximación espacial.

In [ ]:
# Cargar UNGRD y filtrar deslizamientos en Antioquia 2019-2022
df_ungrd = pd.read_csv(_root / "data" / "raw" / "ungrd_emergencias.csv")

landslide_kws = cfg.eventos.landslide_keywords
pattern = "|".join(landslide_kws)

mask = (
    (df_ungrd["departamento"].str.upper() == "ANTIOQUIA") &
    (df_ungrd["evento"].str.lower().str.contains(pattern, na=False))
)
df_ant = df_ungrd[mask].copy()
df_ant["fecha"] = pd.to_datetime(df_ant["fecha"], errors="coerce")
df_ant = df_ant.dropna(subset=["fecha"])
df_ant = df_ant[
    (df_ant["fecha"].dt.year >= cfg.periodo.anio_inicio) &
    (df_ant["fecha"].dt.year <= cfg.periodo.anio_fin)
]

print(f"Eventos deslizamiento Antioquia {cfg.periodo.anio_inicio}-{cfg.periodo.anio_fin}: {len(df_ant)}")
print(f"Tipos de evento: {df_ant['evento'].value_counts().head(5).to_dict()}")

# Geocodificar por centroide municipal
gdf_eventos = get_ungrd_with_coords(df_ant, gdf_mpios)

# Asignar a cuencas por interseccion espacial
gdf_asignado = assign_events_to_cuencas(gdf_eventos, gdf_cuencas)

print(f"\nEventos asignados a cuencas: {len(gdf_asignado)}")
print(f"Cuencas con al menos 1 evento: {gdf_asignado['HYBAS_ID'].nunique()} / {len(gdf_cuencas)}")

### 1.4 Construir grid (semana × cuenca)

El grid es el producto cartesiano de todas las semanas ISO del período
× todas las cuencas de Antioquia, con etiqueta binaria de ocurrencia.

In [ ]:
df_grid = build_event_grid(
    gdf_asignado,
    gdf_cuencas,
    anio_inicio=cfg.periodo.anio_inicio,
    anio_fin=cfg.periodo.anio_fin,
)

print(f"Grid construido: {len(df_grid):,} instancias")
print(f"  Semanas: {df_grid['anio_semana'].nunique()}")
print(f"  Cuencas: {df_grid['HYBAS_ID'].nunique()}")
print(f"  Positivos (deslizamiento=1): {df_grid['deslizamiento'].sum():,} ({df_grid['deslizamiento'].mean():.2%})")
print(f"  Negativos (deslizamiento=0): {(df_grid['deslizamiento']==0).sum():,} ({(df_grid['deslizamiento']==0).mean():.2%})")
print(f"  Ratio desbalance: 1:{(df_grid['deslizamiento']==0).sum()/df_grid['deslizamiento'].sum():.0f}")

### 1.5 Pseudo-ausencias defensibles

Con 0.4% de positivos (1:250), la mayoría de los negativos son confiables.
Aplicamos pseudo-ausencias como filtro adicional para reducir el riesgo de
falsos negativos en las semanas de lluvia intensa o cuencas grandes.

In [ ]:
# Para calcular el umbral de precipitacion necesitamos datos semanales.
# Si CHIRPS esta disponible, usamos su acumulado; si no, calculamos un proxy
# con el numero de eventos de lluvia de IDEAM.
chirps_path = _root / "data" / "raw" / "chirps_antioquia_daily.csv"

if chirps_path.exists():
    from experiment.process import aggregate_weekly_chirps
    df_chirps_daily  = pd.read_csv(chirps_path, parse_dates=["fecha"])
    df_chirps_semanal = aggregate_weekly_chirps(df_chirps_daily)
    df_precip_para_pa = df_chirps_semanal[["anio_semana", "precip_acum_14d"]]
    print(f"CHIRPS disponible: {len(df_chirps_daily)} dias")
else:
    # Fallback: calcular proxy desde el conteo de eventos de lluvia en el grid
    # Usamos n_eventos como proxy de actividad pluviometrica (sesgado, pero funcional)
    print("CHIRPS aun no disponible — usando proxy de eventos como precipitacion")
    df_precip_para_pa = (
        df_grid.groupby("anio_semana")["n_eventos"]
        .sum()
        .reset_index()
        .rename(columns={"n_eventos": "precip_acum_14d"})
    )

df_grid_pa = generate_pseudo_absences(
    df_grid,
    df_precip_para_pa,
    gdf_cuencas,
    precip_percentil=cfg.espacial.pseudo_absence.precip_percentil,
    area_percentil=cfg.espacial.pseudo_absence.area_percentil,
)
print(f"\nDataset con pseudo-ausencias: {len(df_grid_pa):,} instancias")

### 1.6 Construir dataset de modelado

Añadimos features estáticas de cuenca (HydroSHEDS) y codificaciones cíclicas de temporalidad.
Si CHIRPS está disponible, también incluimos las features de precipitación.

In [ ]:
# Features estaticas de cuenca (HydroSHEDS)
static_cols = [c for c in ["HYBAS_ID", "SUB_AREA", "UP_AREA", "DIST_MAIN", "ORDER", "ORDER_"]
               if c in gdf_cuencas.columns]
df_static = pd.DataFrame(gdf_cuencas[static_cols])

df_model = df_grid_pa.merge(df_static, on="HYBAS_ID", how="left")

# Convertir anio_semana a valores numericos para calcular ciclicidad
df_model["semana_num"] = df_model["anio_semana"].apply(lambda p: p.week)
df_model["mes_num"]    = df_model["anio_semana"].apply(lambda p: p.start_time.month)

df_model["semana_sin"] = np.sin(2 * np.pi * df_model["semana_num"] / 52)
df_model["semana_cos"] = np.cos(2 * np.pi * df_model["semana_num"] / 52)
df_model["mes_sin"]    = np.sin(2 * np.pi * df_model["mes_num"] / 12)
df_model["mes_cos"]    = np.cos(2 * np.pi * df_model["mes_num"] / 12)

# Agregar CHIRPS si esta disponible
FEATURES_ESPACIALES = [
    "SUB_AREA", "UP_AREA", "DIST_MAIN",
    "semana_sin", "semana_cos", "mes_sin", "mes_cos",
]
FEATURES_CHIRPS = ["precip_acum_14d", "precip_max_diario_14d",
                   "precip_dias_lluvia_14d", "precip_acum_7d", "precip_acum_3d"]

if chirps_path.exists():
    df_model = df_model.merge(
        df_chirps_semanal[["anio_semana"] + FEATURES_CHIRPS],
        on="anio_semana", how="left",
    )
    FEATURES = FEATURES_ESPACIALES + FEATURES_CHIRPS
    print("Usando features espaciales + CHIRPS")
else:
    FEATURES = FEATURES_ESPACIALES
    print("Usando solo features espaciales (CHIRPS pendiente de descarga)")

FEATURES = [f for f in FEATURES if f in df_model.columns]
TARGET   = cfg.target.nombre

# Ordenar por tiempo para CV correcto
df_model = df_model.sort_values("anio_semana").reset_index(drop=True)

X = df_model[FEATURES]
y = df_model[TARGET]

print(f"\nDataset final: {X.shape}")
print(f"  Features ({len(FEATURES)}): {FEATURES}")
print(f"  Clase 1 (deslizamiento): {y.sum():,} ({y.mean():.2%})")
print(f"  Clase 0 (sin evento)   : {(y==0).sum():,} ({(y==0).mean():.2%})")
print(f"  NaN en X               : {X.isnull().sum().sum()}")

### 1.7 Exploración del dataset

Visualización de la distribución espacial y temporal de eventos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribucion de positivos por cuenca
eventos_por_cuenca = df_model.groupby("HYBAS_ID")[TARGET].sum().sort_values(ascending=False)
top_cuencas = eventos_por_cuenca[eventos_por_cuenca > 0]
axes[0].bar(range(len(top_cuencas)), top_cuencas.values, color="#e74c3c", alpha=0.8)
axes[0].set_title(f"Eventos por cuenca ({len(top_cuencas)} activas / {len(gdf_cuencas)} total)")
axes[0].set_xlabel("Cuenca (ordenada por eventos)")
axes[0].set_ylabel("N eventos")

# 2. Distribucion temporal de positivos
eventos_por_semana = df_model.groupby("anio_semana")[TARGET].sum()
x_pos = range(len(eventos_por_semana))
axes[1].bar(x_pos, eventos_por_semana.values, color="#3498db", alpha=0.8, width=0.9)
axes[1].set_title("Eventos por semana (Antioquia agregado)")
axes[1].set_xlabel("Semana ISO")
axes[1].set_ylabel("N cuencas con evento")
# Marcas en los ejes X cada 52 semanas (anios)
for anio in [2019, 2020, 2021, 2022]:
    idx = next((i for i, p in enumerate(eventos_por_semana.index) if p.start_time.year == anio), None)
    if idx is not None:
        axes[1].axvline(idx, color="gray", linestyle="--", alpha=0.5)
        axes[1].text(idx+1, axes[1].get_ylim()[1]*0.9, str(anio), fontsize=9)

# 3. Distribucion de SUB_AREA y UP_AREA
cuencas_activas = gdf_cuencas[gdf_cuencas["HYBAS_ID"].isin(top_cuencas.index)]
cuencas_inactivas = gdf_cuencas[~gdf_cuencas["HYBAS_ID"].isin(top_cuencas.index)]
axes[2].hist(cuencas_inactivas["SUB_AREA"], bins=30, alpha=0.6, label="Sin evento", color="#95a5a6")
axes[2].hist(cuencas_activas["SUB_AREA"], bins=20, alpha=0.8, label="Con evento", color="#e74c3c")
axes[2].set_title("Distribucion SUB_AREA por actividad")
axes[2].set_xlabel("Area subcuenca (km2)")
axes[2].set_ylabel("N cuencas")
axes[2].legend()

plt.suptitle("Dataset v3 — Antioquia (semana x cuenca)", fontsize=13)
plt.tight_layout()
plt.show()

print(f"SUB_AREA mediana cuencas activas: {cuencas_activas['SUB_AREA'].median():.1f} km2")
print(f"SUB_AREA mediana cuencas inactivas: {cuencas_inactivas['SUB_AREA'].median():.1f} km2")

---
## Pilar 2 — Experimentación con validación temporal de panel

### Estrategia de CV para datos de panel

Con datos de panel `(semana, cuenca)`, el TimeSeriesSplit estándar no funciona directamente
porque los datos no están ordenados por tiempo para todos los índices.

**Estrategia:** splitear sobre las semanas únicas (cronológicamente), y expandir
a todas las cuencas de cada semana. Esto garantiza:
1. El conjunto de entrenamiento siempre termina antes que el de validación (sin leakage)
2. Todas las cuencas de una semana van al mismo fold

```
Semanas totales: ~209
Fold 1: entrenar semanas 1-42  → validar semanas 43-83
Fold 2: entrenar semanas 1-83  → validar semanas 84-125
Fold 3: entrenar semanas 1-125 → validar semanas 126-167
Fold 4: entrenar semanas 1-167 → validar semanas 168-209
```

### Modelos a evaluar

| Modelo | Tipo | Manejo del desbalance |
|--------|------|----------------------|
| `baseline_dummy` | Baseline estocástico | — |
| `logistic_regression` | Lineal | `class_weight='balanced'` |
| `random_forest` | Ensemble árboles | `class_weight='balanced'` |
| `bagging_pu` | PU-Learning | Bootstraps balanceados |

El `BaggingPuClassifier` es el candidato principal: trata clase 0 como 'unlabeled'
y entrena con bootstraps que siempre incluyen todos los positivos + submuestra aleatoria
de negativos. Referencia: Mordelet & Vert (2014), *Machine Learning*.

### 2.1 Función de CV adaptada para panel

In [ ]:
def panel_time_splits(df: pd.DataFrame, n_splits: int = 4):
    """
    Genera indices de train/val cronologicos para datos de panel (semana x cuenca).

    A diferencia de TimeSeriesSplit sobre filas, este splitter opera sobre
    semanas unicas y expande a todas las filas de cada semana.
    Garantiza que no hay ninguna semana de validacion que aparezca en train.
    """
    semanas_unicas = sorted(df["anio_semana"].unique())
    n_sem = len(semanas_unicas)
    tss = TimeSeriesSplit(n_splits=n_splits)

    for train_idx, val_idx in tss.split(range(n_sem)):
        train_semanas = {semanas_unicas[i] for i in train_idx}
        val_semanas   = {semanas_unicas[i] for i in val_idx}

        train_rows = df.index[df["anio_semana"].isin(train_semanas)].tolist()
        val_rows   = df.index[df["anio_semana"].isin(val_semanas)].tolist()
        yield train_rows, val_rows


def evaluar_con_panel_cv(
    pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    df_ref: pd.DataFrame,
    n_splits: int = 4,
) -> dict:
    """
    Evalua un pipeline con CV de panel y retorna metricas promediadas.

    Para BaggingPuClassifier usamos predict_proba para AUC.
    Para modelos que no tienen predict_proba, fallamos graciosamente.
    """
    metricas = {"auc_roc": [], "f1": [], "precision": [], "recall": []}

    for fold, (train_idx, val_idx) in enumerate(
        panel_time_splits(df_ref, n_splits=n_splits), 1
    ):
        X_tr, y_tr = X.loc[train_idx], y.loc[train_idx]
        X_va, y_va = X.loc[val_idx],   y.loc[val_idx]

        # Saltar fold si hay < 2 clases en train o val
        if y_tr.nunique() < 2 or y_va.nunique() < 2:
            print(f"  Fold {fold}: saltado (clase unica)")
            continue

        try:
            pipeline.fit(X_tr, y_tr)

            if hasattr(pipeline, "predict_proba"):
                y_proba = pipeline.predict_proba(X_va)[:, 1]
                auc = roc_auc_score(y_va, y_proba)
            else:
                y_pred_fold = pipeline.predict(X_va)
                auc = roc_auc_score(y_va, y_pred_fold)

            y_pred = pipeline.predict(X_va)
            metricas["auc_roc"].append(auc)
            metricas["f1"].append(f1_score(y_va, y_pred, zero_division=0))
            metricas["precision"].append(precision_score(y_va, y_pred, zero_division=0))
            metricas["recall"].append(recall_score(y_va, y_pred, zero_division=0))

        except Exception as e:
            print(f"  Fold {fold} error: {e}")

    result = {}
    for k, vals in metricas.items():
        arr = np.array(vals) if vals else np.array([0.0])
        result[f"{k}_mean"] = float(arr.mean())
        result[f"{k}_std"]  = float(arr.std())
    return result


# Diagnostico del CV
print("Diagnostico CV de panel:")
for fold, (tr, va) in enumerate(panel_time_splits(df_model, n_splits=4), 1):
    y_tr_f = y.loc[tr]
    y_va_f = y.loc[va]
    sem_tr = df_model.loc[tr, "anio_semana"]
    sem_va = df_model.loc[va, "anio_semana"]
    print(
        f"  Fold {fold}: train {sem_tr.min()}→{sem_tr.max()} "
        f"(pos={y_tr_f.sum()}) | "
        f"val {sem_va.min()}→{sem_va.max()} "
        f"(pos={y_va_f.sum()})"
    )

### 2.2 Definir modelos incluyendo BaggingPuClassifier

In [ ]:
from pulearn import BaggingPuClassifier

n_positivos = int(y.sum())

def make_pipeline(classifier) -> Pipeline:
    """
    Pipeline con imputacion + escalado + clasificador.
    Evita data leakage: imputacion y escalado se ajustan solo con train.
    """
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("clf",     classifier),
    ])


EXPERIMENTOS = {
    # Baseline — referencia minima obligatoria
    "baseline_dummy": DummyClassifier(strategy="stratified", random_state=42),

    # Regresion Logistica — modelo lineal interpretable
    "logistic_regression": LogisticRegression(
        C=1.0,
        class_weight=cfg.target.class_weight,
        max_iter=500,
        random_state=42,
    ),

    # Random Forest — robusto a features correlacionadas
    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        class_weight=cfg.target.class_weight,
        random_state=42,
        n_jobs=-1,
    ),

    # BaggingPuClassifier — PU-Learning (Mordelet & Vert, 2014)
    # Trata clase 0 como 'unlabeled', no como 'confirmed negative'
    # Cada bootstrap incluye todos los positivos + submuestra de negativos
    "bagging_pu": BaggingPuClassifier(
        estimator=RandomForestClassifier(
            n_estimators=50,
            max_depth=6,
            random_state=42,
        ),
        n_estimators=15,
        max_samples=n_positivos,
        n_jobs=-1,
        random_state=42,
    ),
}

print(f"{len(EXPERIMENTOS)} modelos definidos:")
for nombre in EXPERIMENTOS:
    print(f"  - {nombre}")
print(f"\nBaggingPU: n_positivos={n_positivos}, max_samples={n_positivos}")

### 2.3 Ejecutar experimentos y registrar en MLflow

Cada modelo genera un run independiente con sus métricas y el pipeline serializado.
El tag `granularidad=cuenca` permite filtrar estos runs en la UI de MLflow.

In [ ]:
N_SPLITS_CV = 4
resultados_todos = {}

tags_v3 = {
    **cfg.mlflow.run_tags,
    "granularidad": "cuenca",
    "n_cuencas": str(len(gdf_cuencas)),
    "hydrobasins_nivel": str(cfg.espacial.hydrobasins_nivel),
    "fuente_eventos": "ungrd_geocodificado",
    "cv_estrategia": "panel_time_split",
    "pu_learning": "false",
}

for nombre, clasificador in EXPERIMENTOS.items():
    pipeline = make_pipeline(clasificador)
    es_pu = "bagging_pu" in nombre

    run_tags = {**tags_v3, "pu_learning": str(es_pu).lower()}

    with mlflow.start_run(run_name=f"v3_{nombre}", tags=run_tags) as run:

        # Parametros
        params_run = {
            **cfg.as_mlflow_params(),
            "algoritmo": nombre,
            "cv_n_splits": N_SPLITS_CV,
            "n_features": len(FEATURES),
            "n_instancias": len(X),
            "n_positivos": int(y.sum()),
            "pct_positivos": f"{y.mean():.4f}",
        }
        params_run.update({
            f"clf__{k}": str(v)
            for k, v in clasificador.get_params().items()
        })
        mlflow.log_params(params_run)

        # Evaluacion con CV de panel
        print(f"  Evaluando {nombre}...")
        metricas = evaluar_con_panel_cv(pipeline, X, y, df_model, n_splits=N_SPLITS_CV)

        # Registrar metricas
        mlflow.log_metrics(metricas)

        # Entrenar en todo el dataset y serializar
        pipeline.fit(X, y)
        try:
            signature = infer_signature(X, pipeline.predict(X))
            mlflow.sklearn.log_model(pipeline, name="model", signature=signature)
        except Exception:
            mlflow.sklearn.log_model(pipeline, name="model")

        resultados_todos[nombre] = {**metricas, "run_id": run.info.run_id}

    print(
        f"  [{nombre:20s}] "
        f"AUC={metricas['auc_roc_mean']:.3f}+/-{metricas['auc_roc_std']:.3f}  "
        f"Recall={metricas['recall_mean']:.3f}+/-{metricas['recall_std']:.3f}"
    )

print(f"\n{len(EXPERIMENTOS)} runs registrados en MLflow")

### 2.4 Comparación de resultados

In [ ]:
df_resultados = pd.DataFrame(resultados_todos).T
df_resultados = df_resultados.drop(columns=["run_id"])
df_resultados = df_resultados.astype(float).round(4)
df_resultados = df_resultados.sort_values("auc_roc_mean", ascending=False)

print("Resultados v3 — CV de panel (semana x cuenca):")
print("(ordenado por AUC-ROC)\n")
print(df_resultados[[
    "auc_roc_mean", "auc_roc_std",
    "recall_mean",  "recall_std",
    "f1_mean",      "f1_std",
]].to_string())

mejor_modelo = df_resultados.index[0]
print(f"\nMejor modelo: '{mejor_modelo}'")
print(f"  AUC-ROC : {df_resultados.loc[mejor_modelo, 'auc_roc_mean']:.3f}")
print(f"  Recall  : {df_resultados.loc[mejor_modelo, 'recall_mean']:.3f}")

In [ ]:
artifacts_dir = _root / "mlruns" / "artifacts_tmp"
artifacts_dir.mkdir(parents=True, exist_ok=True)
grafica_path  = artifacts_dir / "comparacion_v3_cuencas.png"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
modelos = df_resultados.index.tolist()
colores = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12", "#9b59b6"][:len(modelos)]

auc_means = df_resultados["auc_roc_mean"].values
auc_stds  = df_resultados["auc_roc_std"].values
bars = axes[0].barh(modelos, auc_means, xerr=auc_stds,
                    color=colores, edgecolor="white", capsize=4)
axes[0].axvline(0.5, color="gray", linestyle="--", linewidth=1, label="Azar (0.5)")
axes[0].set_title("AUC-ROC (CV panel)")
axes[0].set_xlabel("AUC-ROC")
axes[0].legend()
axes[0].set_xlim(0, 1)
for bar, val in zip(bars, auc_means):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{val:.3f}", va="center", fontsize=10)

rec_means = df_resultados["recall_mean"].values
rec_stds  = df_resultados["recall_std"].values
bars2 = axes[1].barh(modelos, rec_means, xerr=rec_stds,
                     color=colores, edgecolor="white", capsize=4)
axes[1].set_title("Recall clase positiva (CV panel)")
axes[1].set_xlabel("Recall")
axes[1].set_xlim(0, 1)
for bar, val in zip(bars2, rec_means):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{val:.3f}", va="center", fontsize=10)

plt.suptitle("Pipeline v3 — Granularidad Cuenca (semana x cuenca)", fontsize=13)
plt.tight_layout()
fig.savefig(grafica_path, dpi=120, bbox_inches="tight")
plt.show()

# Registrar grafica en el mejor run
with mlflow.start_run(run_id=resultados_todos[mejor_modelo]["run_id"]):
    mlflow.log_artifact(str(grafica_path))

print(f"Grafica guardada: {grafica_path.relative_to(_root)}")

---
## Pilar 3 — Model Registry

### 3.1 Registrar el mejor modelo

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=cfg.mlflow_tracking_uri)

MODEL_REGISTRY_NAME = f"{cfg.geo.departamento}_deslizamiento_v3_cuenca"

best_run_id = resultados_todos[mejor_modelo]["run_id"]
model_uri   = f"runs:/{best_run_id}/model"

result = mlflow.register_model(model_uri=model_uri, name=MODEL_REGISTRY_NAME)

print(f"Modelo registrado en el registry")
print(f"  Nombre   : {MODEL_REGISTRY_NAME}")
print(f"  Version  : {result.version}")
print(f"  Run ID   : {best_run_id}")
print(f"  Algoritmo: {mejor_modelo}")

### 3.2 Tags de trazabilidad y transición a Staging

In [ ]:
for key, value in [
    ("algoritmo",         mejor_modelo),
    ("auc_roc_cv",        f"{df_resultados.loc[mejor_modelo, 'auc_roc_mean']:.4f}"),
    ("recall_cv",         f"{df_resultados.loc[mejor_modelo, 'recall_mean']:.4f}"),
    ("dataset_version",   "v3"),
    ("granularidad",      "cuenca"),
    ("n_cuencas",         str(len(gdf_cuencas))),
    ("hydrobasins_nivel", str(cfg.espacial.hydrobasins_nivel)),
    ("entrenado_con",     f"{cfg.geo.departamento} {cfg.periodo.anio_inicio}-{cfg.periodo.anio_fin}"),
]:
    client.set_model_version_tag(
        name=MODEL_REGISTRY_NAME, version=result.version, key=key, value=value
    )

print(f"Tags agregados a version {result.version}")

AUC_MINIMO_STAGING = 0.60  # umbral mas bajo que v2 — dataset diferente
auc_del_mejor = df_resultados.loc[mejor_modelo, "auc_roc_mean"]

if auc_del_mejor >= AUC_MINIMO_STAGING:
    client.transition_model_version_stage(
        name=MODEL_REGISTRY_NAME,
        version=result.version,
        stage="Staging",
        archive_existing_versions=False,
    )
    print(f"Version {result.version} promovida a Staging")
    print(f"  AUC-ROC ({auc_del_mejor:.3f}) >= umbral ({AUC_MINIMO_STAGING})")
else:
    print(f"AUC-ROC ({auc_del_mejor:.3f}) < umbral ({AUC_MINIMO_STAGING}) — no promovido")
    print(f"  Accion: ejecutar nuevamente cuando CHIRPS este disponible")

### 3.3 Comparación v2 (departamental) vs v3 (cuenca)

In [ ]:
# Buscar runs de v2 (departamental) para comparar
runs_v2 = mlflow.search_runs(
    experiment_names=[cfg.mlflow.experiment_name],
    filter_string="tags.granularidad IS NULL",
    order_by=["metrics.auc_roc_mean DESC"],
    max_results=1,
)

print("Comparacion v2 vs v3:")
print("=" * 55)

if not runs_v2.empty:
    auc_v2 = runs_v2.iloc[0].get("metrics.auc_roc_mean", 0.0)
    print(f"  v2 (departamental) AUC: {auc_v2:.3f}")
else:
    auc_v2 = 0.552  # valor conocido del Notebook 02
    print(f"  v2 (departamental) AUC: {auc_v2:.3f} (valor de referencia Notebook 02)")

auc_v3 = df_resultados.loc[mejor_modelo, "auc_roc_mean"]
print(f"  v3 (cuenca) AUC       : {auc_v3:.3f}")
print(f"  Mejora absoluta       : {auc_v3 - auc_v2:+.3f}")
print()

# Nota interpretativa importante
print("NOTA: AUC no es directamente comparable entre v2 y v3:")
print("  - v2: 204 instancias, 64% positivos (facil trivialmente)")
print("  - v3: ~25k instancias con pseudo-ausencias, 0.4% positivos (PU-real)")
print("  Un AUC bajo en v3 con datos reales puede ser mas valioso que")
print("  un AUC alto en v2 con etiquetas artificialmente infladas.")

In [ ]:
print("=" * 60)
print("RESUMEN — PIPELINE v3 CUENCAS")
print("=" * 60)
print(f"  Experimento MLflow    : {cfg.mlflow.experiment_name}")
print(f"  Granularidad          : cuenca (HydroSHEDS nivel {cfg.espacial.hydrobasins_nivel})")
print(f"  Cuencas Antioquia     : {len(gdf_cuencas)}")
print(f"  Municipios geocodif.  : {len(gdf_mpios)} (match UNGRD: ~96.6%)")
print(f"  Instancias del model  : {len(X):,}")
print(f"  Positivos             : {y.sum():,} ({y.mean():.2%})")
print(f"  Features utilizadas   : {len(FEATURES)} ({FEATURES})")
print(f"  Algoritmos evaluados  : {len(EXPERIMENTOS)}")
print(f"  Mejor modelo          : {mejor_modelo}")
print(f"  AUC-ROC (CV panel)    : {df_resultados.loc[mejor_modelo, 'auc_roc_mean']:.3f}")
print(f"  Recall  (CV panel)    : {df_resultados.loc[mejor_modelo, 'recall_mean']:.3f}")
print(f"  Registry              : {MODEL_REGISTRY_NAME} v{result.version}")
print()
print("Proximos pasos:")
print("  1. Esperar fin de descarga CHIRPS y re-ejecutar con features de precipitacion")
print("  2. Descargar ERA5-Land (humedad suelo) cuando esten las credenciales ~/.cdsapirc")
print("  3. Comparar BaggingPU vs RandomForest con CHIRPS disponible")
print("=" * 60)